# 12 · Limpieza: Nuevos Afiliados AGO-SEP_anonimizado

Nuevas empresas afiliadas en agosto y septiembre, con su afiliado principal. Una fila = un nuevo afiliado.

- Entrada: `datos/originales/Nuevos Afiliados AGO-SEP_anonimizado.xlsx` (hoja `Hoja1`)
- Salida: `datos/limpios/12_nuevos_afiliados_limpio.csv`

In [ ]:
from pathlib import Path

import pandas as pd

BASE = Path.cwd().parent
ORIGINALES = BASE / "datos" / "originales"
LIMPIOS = BASE / "datos" / "limpios"
LIMPIOS.mkdir(exist_ok=True)

## 1. Cómo viene la hoja
Dos bloques, uno por mes, uno debajo del otro. Cada bloque tiene: `Nuevos afiliados`, el nombre del mes (`Agosto`), una fila de encabezados y las empresas.
Correo y contacto vienen vacíos por la anonimización.

**Cómo se limpia:** cada empresa toma el mes del último nombre de mes que apareció arriba (`ffill`) y se dejan solo las filas con id de empresa (`EMP-0000`).
Así quedan fuera, en un solo paso, títulos, encabezados repetidos y filas vacías.

El archivo no dice el año. Se toma **2026**, porque el nombre del archivo es "AGO-SEP" y los demás reportes comerciales son de 2026.

In [ ]:
crudo = pd.read_excel(ORIGINALES / "Nuevos Afiliados AGO-SEP_anonimizado.xlsx", sheet_name="Hoja1", header=None)
assert crudo.shape == (18, 4), crudo.shape
assert (crudo[0] == "NOMBRE EMPRESA [anonimizado]").sum() == 2 and (crudo[1] == "AFILIADO PRINCIPAL [anonimizado]").sum() == 2

MESES = {"enero": 1, "febrero": 2, "marzo": 3, "abril": 4, "mayo": 5, "junio": 6, "julio": 7,
         "agosto": 8, "septiembre": 9, "octubre": 10, "noviembre": 11, "diciembre": 12}
ANIO = 2026

col0 = crudo[0].astype("string").str.strip()
crudo["mes"] = col0.str.lower().map(MESES).ffill()
print("Meses encontrados:", col0[col0.str.lower().isin(MESES)].tolist())

df = crudo.loc[col0.str.fullmatch(r"EMP-\d{4}", na=False), ["mes", 0, 1]].copy()
df.columns = ["mes_afiliacion", "id_empresa", "id_persona"]
df["mes_afiliacion"] = pd.to_datetime(dict(year=ANIO, month=df["mes_afiliacion"].astype(int), day=1))
df["id_empresa"] = df["id_empresa"].str.strip()
df["id_persona"] = df["id_persona"].str.strip()
df = df.reset_index(drop=True)
df

## 2. Validaciones
- Todas las filas tienen mes, empresa y afiliado principal.
- Cada empresa aparece una sola vez y los ids tienen el formato de las demás bases.
- 6 afiliados en agosto y 4 en septiembre, como en el archivo.

In [ ]:
assert df.notna().all().all()
assert df["id_empresa"].is_unique
assert df["id_empresa"].str.fullmatch(r"EMP-\d{4}").all() and df["id_persona"].str.fullmatch(r"P-\d{5}").all()
conteo = df["mes_afiliacion"].dt.month.value_counts().to_dict()
assert conteo == {8: 6, 9: 4}, conteo

## 3. Guardar
`utf-8-sig` para que Excel y Looker Studio lean bien las tildes. Comparte `id_empresa` con las demás bases e `id_persona` con el 05 (afiliados).

In [ ]:
salida = LIMPIOS / "12_nuevos_afiliados_limpio.csv"
df.to_csv(salida, index=False, encoding="utf-8-sig")

assert pd.read_csv(salida).shape == (10, 3)
salida.name